# Cascade

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/grounded-ai/grounded_ai/blob/main/examples/cascade.ipynb)

An LLM judge is accurate but slow and expensive. A decision model answers typed questions in one cheap request and reports how sure it is, measured from its probabilities. `CascadeEvaluator` asks the decision model first and sends only the answers below `min_confidence` to the judge. On 200 real agent turns this cut the bill 61% against Claude Sonnet 4.6 alone, with no accuracy lost ([benchmark](https://github.com/grounded-ai/grounded_ai/blob/main/benchmarks/tool_call/bench.py)).

This notebook runs the same three checks through a cascade with each of three decision models, with Claude Haiku 4.5 as the judge, then compares the three on labelled data:

1. **agent-decision-3b**, our open model, served next to the notebook (no key)
2. **Jev**, TypeSafe's hosted model (`TYPESAFE_API_KEY`)
3. **OpenAI Decisions** (`OPENAI_API_KEY`)

## Setup

**In Colab:** add your keys under Secrets (the key icon on the left) and give this notebook access: `ANTHROPIC_API_KEY` for the judge, plus `TYPESAFE_API_KEY` and `OPENAI_API_KEY` for the hosted decision models you want to try. A section whose key is missing is skipped. Use a GPU runtime (Runtime > Change runtime type > T4 GPU, free) for agent-decision-3b.

**Elsewhere:** `pip install "grounded-ai[jev]"` and set the same keys as environment variables.

In [1]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q "grounded-ai[jev]>=2.1.0"
    from google.colab import userdata

    for key in ("ANTHROPIC_API_KEY", "TYPESAFE_API_KEY", "OPENAI_API_KEY"):
        try:
            os.environ[key] = userdata.get(key)
        except Exception:  # not set, or the notebook has no access to it
            pass
print("keys:", [key for key in ("ANTHROPIC_API_KEY", "TYPESAFE_API_KEY", "OPENAI_API_KEY") if os.getenv(key)])

keys: ['ANTHROPIC_API_KEY', 'TYPESAFE_API_KEY', 'OPENAI_API_KEY']


In Colab, this cell serves agent-decision-3b at `/v1/systemone`, the API decision models share (the first run downloads about 7 GB). Elsewhere, serve it with `python serve/decision_server.py` from the [agent-decision repo](https://github.com/grounded-ai/agent-decision) and set `AGENT_DECISION_URL=http://127.0.0.1:8765`.

In [2]:
import json
import subprocess
import time
import urllib.request

if IN_COLAB and not os.getenv("AGENT_DECISION_URL"):
    subprocess.run(["git", "clone", "-q", "https://github.com/grounded-ai/agent-decision", "/content/agent-decision"])
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=5.10.2", "peft>=0.18"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])  # Colab's torchao is too old for transformers 5
    server = subprocess.Popen([sys.executable, "/content/agent-decision/serve/decision_server.py"],
                              stdout=open("/content/agent_decision.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(180):
        try:
            health = json.load(urllib.request.urlopen("http://127.0.0.1:8765/health", timeout=5))
            os.environ["AGENT_DECISION_URL"] = "http://127.0.0.1:8765"
            print("agent-decision-3b is up on", health["device"])
            break
        except Exception:
            if server.poll() is not None:
                print(open("/content/agent_decision.log").read()[-2000:])
                break
            time.sleep(10)

agent-decision-3b is up on cuda


## The checks

The same three go through every cascade:

- **hallucination**: does a response contradict its context?
- **tool call**: should an agent's proposed tool call run, and if not, what is wrong with it? (The questions are worded as agent-decision was trained.)
- **support ticket**: what should a support agent do next, and how is the customer feeling? Two questions in one request; only the ones the decision model is unsure about go to the judge, together, in one call.

`run_checks` prints who answered each question: the decision model with its confidence, or the judge with its reasoning.

In [3]:
from grounded_ai import CascadeEvaluator, confidence
from grounded_ai.backends.jev import HALLUCINATION, ChoiceQuestion, JevInput, NoulAnswer, NoulQuestion
from grounded_ai.schemas import EvaluationError

JUDGE = "anthropic/claude-haiku-4-5"

CHECKS = {
    "hallucination": JevInput(
        state={"context": "Apollo 11 landed on the Moon in 1969. Neil Armstrong and Buzz Aldrin walked on the "
                          "surface while Michael Collins stayed in orbit.",
               "response": "Buzz Aldrin stayed in orbit while Collins walked on the Moon."},
        questions={"verdict": HALLUCINATION},
    ),
    "tool call": JevInput(
        state=(
            'TOOLS:\n{"name": "iplookup", "description": "Look up the location and owner of an IP address.", '
            '"parameters": {"ip": {"description": "The IP address to look up.", "type": "str"}}}\n\n'
            "USER: Lookup the details of the IP address 8.8.8.8.\n\n"
            'ASSISTANT (tool calls):\n-> iplookup({"ip": "185.132.189.11"})'
        ),
        questions={
            "run_it": NoulQuestion(instructions="Should these tool calls be executed exactly as proposed?"),
            "problem": ChoiceQuestion(
                instructions="What, if anything, is wrong with the proposed tool calls?",
                criteria={
                    "correct": "the calls are right and do everything the user asked",
                    "wrong_tool": "a call uses a tool that cannot do what was asked",
                    "wrong_arguments": "the right tool is called with arguments that do not match the request or the tool",
                    "incomplete": "the calls are right but part of the request is not done",
                },
            ),
        },
    ),
    "support ticket": JevInput(
        state="Can you also cc my manager on the refund? Her email is in the thread.",
        questions={
            "action": ChoiceQuestion(
                instructions="What should the support agent do next?",
                criteria={"call_tool": "use a tool (email, refund system)", "respond": "reply to the customer",
                          "ask": "ask the customer for missing details"},
            ),
            "tone": ChoiceQuestion(instructions="Classify the customer's tone.",
                                   criteria={"calm": "neutral or polite", "upset": "angry or frustrated"}),
        },
    ),
}


def pick(answer):
    if isinstance(answer, NoulAnswer):
        return f"{'yes' if answer.noul >= 0.5 else 'no'} (p={answer.noul:.2f})"
    return str(answer.choice if answer.type == "choice" else answer.score)


def run_checks(cascade):
    for check, input_data in CHECKS.items():
        result = cascade.evaluate(input_data)
        if isinstance(result, EvaluationError):
            print(f"{check}: {result.error_code}: {result.message}")
            continue
        print(check)
        for question, answer in result.answers.items():
            first = result.decision.answers[question]
            if question in result.judged:
                print(f"  {question:8} {str(answer.answer):16} judge     (decision model said {pick(first)}, "
                          f"confidence {confidence(first):.2f})")
            else:
                print(f"  {question:8} {pick(answer):16} decision  (confidence {confidence(answer):.2f})")
        for question in result.judged:
            print(f"  judge on {question}: {result.answers[question].reasoning}")
        print()


cascades = {}  # the ones that run, for the comparison at the end

## 1. agent-decision-3b

Our open decision model ([model card](https://huggingface.co/grounded-ai/agent-decision-3b)), trained on agent decisions: tool calls, agent steps, whether to call, ask or decline, and grounding. Any model served at `/v1/systemone` is used the same way: `systemone/<model>` and its `base_url` (or `host="llamacpp"` / `host="ollama"`).

In [4]:
if os.getenv("AGENT_DECISION_URL"):
    cascades["agent-decision-3b"] = CascadeEvaluator(
        decision_model="systemone/agent-decision-3b",
        decision_model_kwargs={"base_url": os.environ["AGENT_DECISION_URL"]},
        judge=JUDGE,
        min_confidence=0.7,
    )
    run_checks(cascades["agent-decision-3b"])
else:
    print("Skipped: agent-decision-3b is not running (see Setup).")

hallucination
  verdict  hallucination    decision  (confidence 0.98)

tool call
  run_it   no (p=0.00)      decision  (confidence 1.00)
  problem  wrong_arguments  decision  (confidence 0.99)


support ticket
  action   ask              judge     (decision model said call_tool, confidence 0.49)
  tone     calm             decision  (confidence 0.97)
  judge on action: The customer is asking the support agent to CC their manager on a refund communication. However, the customer states 'Her email is in the thread' - but from the support agent's perspective (who would be reading this request), they don't have access to view the previous email thread in this isolated message. To properly fulfill this request, the agent needs clarification on which email address to use or needs to retrieve that information. Before taking action (calling a tool to send an email or process a refund), the agent should ask for the manager's email address to be provided explicitly in this conversation, since they cannot reliably access or verify which email in a thread is the correct manager contact.


## 2. Jev

TypeSafe's hosted decision model ([docs](https://docs.typesafe.ai)).

In [5]:
if os.getenv("TYPESAFE_API_KEY"):
    cascades["jev"] = CascadeEvaluator(decision_model="jev/jev-latest", judge=JUDGE, min_confidence=0.7)
    run_checks(cascades["jev"])
else:
    print("Skipped: no TYPESAFE_API_KEY.")

hallucination
  verdict  hallucination    decision  (confidence 1.00)


tool call
  run_it   no (p=0.03)      decision  (confidence 0.94)
  problem  wrong_arguments  decision  (confidence 1.00)


support ticket
  action   call_tool        judge     (decision model said call_tool, confidence 0.23)
  tone     calm             judge     (decision model said calm, confidence 0.66)
  judge on action: The customer is asking for their manager to be CC'd on a refund communication. The customer has provided the manager's email is in the thread. The support agent should take action by calling a tool (email system) to send the refund and CC the manager, rather than responding without action or asking for more details since the information needed has been provided.
  judge on tone: The customer's message is polite and reasonable. They are making a request ('Can you also cc my manager') using courteous language. There is no indication of frustration, anger, or upset tone - the request is straightforward and professional.


## 3. OpenAI Decisions

OpenAI's Decisions API (`gpt-6-luna`). grounded-ai translates the questions and answers, so it takes the same input and gives the same output as the others.

In [6]:
if os.getenv("OPENAI_API_KEY"):
    cascades["openai-decisions"] = CascadeEvaluator(decision_model="openai-decisions/gpt-6-luna", judge=JUDGE,
                                                    min_confidence=0.7)
    run_checks(cascades["openai-decisions"])
else:
    print("Skipped: no OPENAI_API_KEY.")

hallucination
  verdict  hallucination    decision  (confidence 1.00)


tool call
  run_it   no (p=0.00)      decision  (confidence 1.00)
  problem  wrong_arguments  decision  (confidence 1.00)


support ticket
  action   respond          judge     (decision model said respond, confidence 0.19)
  tone     calm             decision  (confidence 1.00)
  judge on action: The customer is asking the support agent to cc their manager on a refund, and mentions that the manager's email is in the thread. To proceed with this request, the agent needs to locate the manager's email address from the thread. Since the email is stated to be in the thread, the agent should have the necessary information to take action. However, the customer hasn't provided explicit confirmation that the refund process has started or what the refund details are. The most appropriate next action would be to respond to the customer, either to confirm receipt of the request and that they will cc the manager, or to clarify refund details before processing. Since the agent needs to actually process this request (cc the manager on a refund action), the agent should likely respond to acknowledge the request and potent

## Compare them on labelled data

Three checks don't tell you which decision model to use, or at what threshold. `compare()` asks each one every labelled item and reports, for each threshold, how much it settles alone (`settled`), how often those answers are right (`right`), and the cascade's accuracy with the judge. With `target_accuracy` it recommends the lowest threshold that meets it. The judge is asked once per item, and only about items some model would send it.

An item is a state, its questions, and the right answer to each (the option for a choice, `True`/`False` for a yes/no, the level index for a score). Here: 30 rows of [`agent-decision-eval`](https://huggingface.co/datasets/grounded-ai/agent-decision-eval), six from each question family. agent-decision was trained on a held-out split of the same sources, so it has a home advantage; use a few hundred items of your own traffic to decide.

In [7]:
import collections
import random

from huggingface_hub import hf_hub_download

from grounded_ai import compare

path = hf_hub_download("grounded-ai/agent-decision-eval", "test.jsonl", repo_type="dataset")
rows = [json.loads(line) for line in open(path)]
by_family = collections.defaultdict(list)
for row in rows:
    families = set(row["families"].values())
    by_family["tool_call" if "tool_call_ok" in families else families.pop()].append(row)
rng = random.Random(7)
items = [{"state": row["state"], "questions": row["questions"], "labels": row["gold"]}
         for family in sorted(by_family) for row in rng.sample(by_family[family], 6)]
print(len(items), "items,", sum(len(item["questions"]) for item in items), "questions")

report = compare(items, {name: cascade.decision_model for name, cascade in cascades.items()}, judge=JUDGE,
                 thresholds=(0.5, 0.7, 0.9), target_accuracy=0.9)
print(report.table())

30 items, 32 questions


model                                          n accuracy    ECE  s/check  tokens  @0.5  settled/right/cascade  @0.7  settled/right/cascade  @0.9  settled/right/cascade
agent-decision-3b                             32    0.969  0.072    0.926    1247          94%/1.00/1.000          84%/1.00/0.969          66%/1.00/0.969
                                             act alone at >= 0.5 for 90% right
jev                                           32    0.938  0.083    0.246    1569          88%/0.96/0.906          88%/0.96/0.906          66%/0.95/0.906
                                             act alone at >= 0.5 for 90% right
openai-decisions                              32    0.906  0.113    0.329    1349          94%/0.93/0.938          84%/0.93/0.938          84%/0.93/0.938
                                             act alone at >= 0.5 for 90% right
judge: anthropic/claude-haiku-4-5, 0.909 right on the items it saw


Everything in the table is also on the report (`report.models[name].thresholds`, `.recommended_threshold`, ...), and `report.model_dump_json()` saves it. From the command line, with items as JSONL:

```bash
grounded-ai compare items.jsonl --model systemone/agent-decision-3b@http://127.0.0.1:8765 \
    --model jev/jev-latest --model openai-decisions/gpt-6-luna --judge anthropic/claude-haiku-4-5 \
    --thresholds 0.5 0.7 0.9 --target-accuracy 0.9
```